In [7]:
import random
import time
from collections import deque
import ipywidgets as widgets
from IPython.display import display, clear_output

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
    return res

def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt
        
    if r > 0: successors.append(("Lên", swap(mt, pos, pos - 3)))
    if r < 2: successors.append(("Xuống", swap(mt, pos, pos + 3)))
    if c > 0: successors.append(("Trái", swap(mt, pos, pos - 1)))
    if c < 2: successors.append(("Phải", swap(mt, pos, pos + 1)))
    return successors

def bfs_early(start_state, goal_state):
    if start_state == goal_state:
        return [], 0
    frontier = deque([(start_state, [])])
    explored = set([tuple(start_state)])
    nodes_generated = 1
    
    while frontier:
        node, path = frontier.popleft()
        
        for action, child in get_successors(node):
            nodes_generated += 1
            if child == goal_state:
                return path + [(action, child)], nodes_generated
                
            if tuple(child) not in explored:
                explored.add(tuple(child))
                frontier.append((child, path + [(action, child)]))
                
    return None, nodes_generated

input_boxes = [widgets.IntText(value=v, layout=widgets.Layout(width='50px', height='50px')) 
               for v in [2, 8, 3, 1, 6, 4, 7, 0, 5]]
grid = widgets.GridBox(input_boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 60px)"))
btn_solve = widgets.Button(description="Giải (Early Goal Test)", button_style='success', layout=widgets.Layout(width='180px'))
output_area = widgets.Output()

def on_solve(b):
    with output_area:
        clear_output()
        start_state = [box.value for box in input_boxes]
        goal_state = [1, 2, 3, 8, 0, 4, 7, 6, 5]
        print("Trạng thái bắt đầu:")
        print(in_mt(start_state))
        print("Đang giải...")
        start_time = time.time()
        
        path, nodes_generated = bfs_early(start_state, goal_state)
        
        end_time = time.time()
        if path is not None:
            print(f"Thành công! Số bước (độ sâu): {len(path)}")
            print(f"Số trạng thái đã sinh: {nodes_generated}")
            print(f"Thời gian giải: {end_time - start_time:.4f} giây")
            for step, (action, state) in enumerate(path):
                print(f"Bước {step + 1}: Di chuyển ô trống sang {action}")
                print(in_mt(state))
        else:
            print("Không tìm thấy giải pháp (hoặc quá thời gian)!")

btn_solve.on_click(on_solve)
display(widgets.VBox([widgets.Label("Nhập ma trận 8-puzzle (từ trái qua phải, từ trên xuống dưới):"), grid, btn_solve, output_area]))